In [1]:
import pandas as pd
import numpy as np
import holidays
import openmeteo_requests
import requests_cache
from retry_requests import retry
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
df=pd.read_csv("Ecommerce_Sales_Data_2024_2025.csv")

In [3]:
df['Order_Date'] = pd.to_datetime(df['Order Date'])


In [4]:
df

,Order ID,Order Date,Customer Name,Region,City,Category,Sub-Category,Product Name,Quantity,Unit Price,Discount,Sales,Profit,Payment Mode,Order_Date
0,10001,2024-10-19,Kashvi Varty,South,Bangalore,Books,Non-Fiction,Non-Fiction Ipsum,2,36294,5,68958.6,10525.09,Debit Card,2024-10-19
1,10002,2025-08-30,Advik Desai,North,Delhi,Groceries,Rice,Rice Nemo,1,42165,20,33732.0,6299.66,Debit Card,2025-08-30
2,10003,2023-11-04,Rhea Kalla,East,Patna,Kitchen,Juicer,Juicer Odio,4,64876,20,207603.2,19850.27,Credit Card,2023-11-04
3,10004,2025-05-23,Anika Sen,East,Kolkata,Groceries,Oil,Oil Doloribus,5,37320,15,158610.0,36311.02,UPI,2025-05-23
4,10005,2025-01-19,Akarsh Kaul,West,Pune,Clothing,Kids Wear,Kids Wear Quo,1,50037,10,45033.3,9050.04,Debit Card,2025-01-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,14996,2024-06-25,Nishith Kulkarni,East,Kolkata,Books,Fiction,Fiction Veritatis,3,60671,0,182013.0,11853.15,Debit Card,2024-06-25
4996,14997,2024-12-22,Aaina Chander,North,Jaipur,Toys,Doll,Doll Nulla,5,70048,0,350240.0,31237.23,Credit Card,2024-12-22
4997,14998,2025-04-15,Dhanush Gara,South,Bangalore,Beauty,Lipstick,Lipstick Eaque,1,42162,15,35837.7,7827.50,Debit Card,2025-04-15
4998,14999,2024-07-08,Divyansh Malhotra,East,Kolkata,Electronics,Smartwatch,Smartwatch Adipisci,4,13568,10,48844.8,6603.86,Credit Card,2024-07-08


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Order ID       5000 non-null   int64         
 1   Order Date     5000 non-null   object        
 2   Customer Name  5000 non-null   object        
 3   Region         5000 non-null   object        
 4   City           5000 non-null   object        
 5   Category       5000 non-null   object        
 6   Sub-Category   5000 non-null   object        
 7   Product Name   5000 non-null   object        
 8   Quantity       5000 non-null   int64         
 9   Unit Price     5000 non-null   int64         
 10  Discount       5000 non-null   int64         
 11  Sales          5000 non-null   float64       
 12  Profit         5000 non-null   float64       
 13  Payment Mode   5000 non-null   object        
 14  Order_Date     5000 non-null   datetime64[ns]
dtypes: datetime64[ns](1),

In [6]:
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Day'] = df['Order_Date'].dt.day
df['DayOfWeek'] = df['Order_Date'].dt.dayofweek
df['IsWeekend'] = df['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
df['Quarter'] = df['Order_Date'].dt.quarter

In [7]:
df['Sales_Lag_1'] = df.groupby(['City', 'Category'])['Sales'].shift(1)

df['Sales_Lag_7'] = df.groupby(['City', 'Category'])['Sales'].shift(7)
df['Sales_Lag_14'] = df.groupby(['City', 'Category'])['Sales'].shift(14)
df['Sales_Lag_30'] = df.groupby(['City', 'Category'])['Sales'].shift(30)

In [8]:
df['Rolling_Avg_7'] = df.groupby(['City', 'Category'])['Sales'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)
df['Rolling_Avg_14'] = df.groupby(['City', 'Category'])['Sales'].transform(
    lambda x: x.rolling(window=14, min_periods=1).mean()
)

In [9]:
df['Discount_Rate'] = df['Discount'] / 100
df['Effective_Price'] = df['Unit Price'] * (1 - df['Discount_Rate'])

# 5. Product Hierarchy Features
df['Category_SubCategory'] = df['Category'] + '_' + df['Sub-Category']

# 6. Customer Features
df['Customer_Count'] = df.groupby(['City', 'Category'])['Customer Name'].transform('count')


In [10]:
df['Sales_Lag_1']=df['Sales_Lag_1'].fillna(df['Sales_Lag_1'].median())
df['Sales_Lag_7']=df['Sales_Lag_7'].fillna(df['Sales_Lag_7'].median())
df['Sales_Lag_14']=df['Sales_Lag_14'].fillna(df['Sales_Lag_14'].median())
df['Sales_Lag_30']=df['Sales_Lag_30'].fillna(df['Sales_Lag_30'].median())
df['Rolling_Avg_7']=df['Rolling_Avg_7'].fillna(df['Rolling_Avg_7'].median())
df['Rolling_Avg_14']=df['Rolling_Avg_14'].fillna(df['Rolling_Avg_14'].median())

In [11]:

india_holidays = holidays.country_holidays('IN', years=[2023, 2024, 2025])

# Function to add holiday features
def add_holiday_features(df):
    
    
    # Is Holiday
    df['Is_Holiday'] = df['Order_Date'].dt.date.apply(
        lambda x: 1 if x in india_holidays else 0
    )
    
    #
    # Holiday Type (e.g., National, Religious)
    df['Holiday_Type'] = df['Order_Date'].dt.date.apply(
        lambda x: india_holidays.get(x, 'None') if x in india_holidays else 'None'
    )
    
    return df

# Apply to your dataframe
df = add_holiday_features(df)

In [12]:
df['Sales_Lag_1'][0]

np.float64(82943.65)

In [13]:

def get_weather_data(city, date, api_key=None):
    """
    Fetch weather data for a specific city and date using OpenMeteo
    """
    # City coordinates (you can get these from geocoding or hardcode for major cities)
    city_coordinates = {
    'Bangalore': (12.9716, 77.5946),
    'Lucknow': (26.8467, 80.9462),
    'Guwahati': (26.1445, 91.7362),
    'Chandigarh': (30.7333, 76.7794),
    'Jaipur': (26.9124, 75.7873),
    'Amritsar': (31.6340, 74.8723),
    'Surat': (21.1702, 72.8311),
    'Patna': (25.5941, 85.1376),
    'Bhubaneswar': (20.2961, 85.8245),
    'Ranchi': (23.3441, 85.3096),
    'Goa': (15.2993, 74.1240),
    'Delhi': (28.6139, 77.2090),
    'Pune': (18.5204, 73.8567),
    'Ahmedabad': (23.0225, 72.5714),
    'Chennai': (13.0827, 80.2707),
    'Kolkata': (22.5726, 88.3639),
    'Mumbai': (19.0760, 72.8777),
    'Hyderabad': (17.3850, 78.4867),
    'Coimbatore': (11.0168, 76.9558),
    'Thiruvananthapuram': (8.5241, 76.9366)
}
    
    if city not in city_coordinates:
        return {'temperature': np.nan, 'rainfall': np.nan}
    
    lat, lon = city_coordinates[city]
    
    # Set up session with caching
    session = requests_cache.CachedSession('weather_cache', expire_after=3600)
    session = retry(session)
    openmeteo = openmeteo_requests.Client(session=session)
    
    # Weather parameters
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date.strftime('2023-01-01'),
        "end_date": date.strftime('2025-12-31'),
        "daily": ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "rain_sum"],
        "timezone": "auto"
    }
    
    try:
        responses = openmeteo.weather_api(url, params=params)
        response = responses[0]
        
        daily = response.Daily()
        temperature_max = daily.Variables(0).ValuesAsNumpy()
        temperature_min = daily.Variables(1).ValuesAsNumpy()
        precipitation = daily.Variables(2).ValuesAsNumpy()
        rain = daily.Variables(3).ValuesAsNumpy()
        
        return {
            'temperature_max': temperature_max[0],
            'temperature_min': temperature_min[0],
            'precipitation_sum': precipitation[0],
            'rain_sum': rain[0]
        }
    except Exception as e:
        print(f"Error fetching weather for {city} on {date}: {e}")
        return {'temperature_max': np.nan, 'temperature_min': np.nan, 'precipitation_sum': np.nan, 'rain_sum': np.nan}

# Apply weather data to your dataframe
def add_weather_features(df):
    df['Order Date'] = pd.to_datetime(df['Order Date'])
    
    weather_data = []
    
    for index, row in df.iterrows():
        weather = get_weather_data(row['City'], row['Order Date'])
        weather_data.append(weather)
    
    weather_df = pd.DataFrame(weather_data)
    df = pd.concat([df, weather_df], axis=1)
    
    return df


df = add_weather_features(df)

In [14]:
df

,Order ID,Order Date,Customer Name,Region,City,Category,Sub-Category,Product Name,Quantity,Unit Price,...,Discount_Rate,Effective_Price,Category_SubCategory,Customer_Count,Is_Holiday,Holiday_Type,temperature_max,temperature_min,precipitation_sum,rain_sum
0,10001,2024-10-19,Kashvi Varty,South,Bangalore,Books,Non-Fiction,Non-Fiction Ipsum,2,36294,...,0.05,34479.3,Books_Non-Fiction,27,0,None,27.450001,14.90,0.0,0.0
1,10002,2025-08-30,Advik Desai,North,Delhi,Groceries,Rice,Rice Nemo,1,42165,...,0.20,33732.0,Groceries_Rice,25,0,None,18.299999,7.85,0.0,0.0
2,10003,2023-11-04,Rhea Kalla,East,Patna,Kitchen,Juicer,Juicer Odio,4,64876,...,0.20,51900.8,Kitchen_Juicer,26,0,None,21.450001,10.70,0.0,0.0
3,10004,2025-05-23,Anika Sen,East,Kolkata,Groceries,Oil,Oil Doloribus,5,37320,...,0.15,31722.0,Groceries_Oil,25,0,None,24.700001,15.75,0.2,0.2
4,10005,2025-01-19,Akarsh Kaul,West,Pune,Clothing,Kids Wear,Kids Wear Quo,1,50037,...,0.10,45033.3,Clothing_Kids Wear,23,0,None,29.850000,14.15,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,14996,2024-06-25,Nishith Kulkarni,East,Kolkata,Books,Fiction,Fiction Veritatis,3,60671,...,0.00,60671.0,Books_Fiction,24,0,None,24.700001,15.75,0.2,0.2
4996,14997,2024-12-22,Aaina Chander,North,Jaipur,Toys,Doll,Doll Nulla,5,70048,...,0.00,70048.0,Toys_Doll,27,0,None,19.650000,7.15,0.0,0.0
4997,14998,2025-04-15,Dhanush Gara,South,Bangalore,Beauty,Lipstick,Lipstick Eaque,1,42162,...,0.15,35837.7,Beauty_Lipstick,23,0,None,27.450001,14.90,0.0,0.0
4998,14999,2024-07-08,Divyansh Malhotra,East,Kolkata,Electronics,Smartwatch,Smartwatch Adipisci,4,13568,...,0.10,12211.2,Electronics_Smartwatch,28,0,None,24.700001,15.75,0.2,0.2


In [15]:

df['Date_City_Key'] = df['Order_Date'].dt.date.astype(str) + '_' + df['City']




print(df.isnull().sum())

# Fill missing weather data with city averages
df['temperature_max'] = df.groupby('City')['temperature_max'].transform(
    lambda x: x.fillna(x.mean())
)
df['rain_sum'] = df.groupby('City')['rain_sum'].transform(
    lambda x: x.fillna(x.mean())
)

Order ID                0
Order Date              0
Customer Name           0
Region                  0
City                    0
Category                0
Sub-Category            0
Product Name            0
Quantity                0
Unit Price              0
Discount                0
Sales                   0
Profit                  0
Payment Mode            0
Order_Date              0
Year                    0
Month                   0
Day                     0
DayOfWeek               0
IsWeekend               0
Quarter                 0
Sales_Lag_1             0
Sales_Lag_7             0
Sales_Lag_14            0
Sales_Lag_30            0
Rolling_Avg_7           0
Rolling_Avg_14          0
Discount_Rate           0
Effective_Price         0
Category_SubCategory    0
Customer_Count          0
Is_Holiday              0
Holiday_Type            0
temperature_max         0
temperature_min         0
precipitation_sum       0
rain_sum                0
Date_City_Key           0
dtype: int64

In [16]:
df['Temperature_Avg'] = (df['temperature_max'] + df['temperature_min']) / 2
df['Temperature_Variance'] = df['temperature_max'] - df['temperature_min']


# Rainfall Features
df['Is_Rainy_Day'] = df['rain_sum'].apply(lambda x: 1 if x > 0 else 0)
df['Rainfall_Category'] = df['rain_sum'].apply(
    lambda x: 'Heavy' if x > 10 else 'Moderate' if x > 5 else 'Light' if x > 0 else 'None'
)

# Holiday + Weather Interaction
df['Holiday_Rainfall'] = df['Is_Holiday'] * df['rain_sum']
df['Holiday_Temperature'] = df['Is_Holiday'] * df['Temperature_Avg']

#Seasonal Features
df['Season'] = df['Order_Date'].dt.month.apply(
    lambda x: 'Summer' if x in [3, 4, 5] else 'Monsoon' if x in [6, 7, 8, 9] 
              else 'Winter' if x in [10, 11, 12, 1, 2] else 'Spring'
)

# Weekday + Holiday Interaction
df['Weekend_Holiday'] = df['IsWeekend'] * df['Is_Holiday']

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 46 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Order ID              5000 non-null   int64         
 1   Order Date            5000 non-null   datetime64[ns]
 2   Customer Name         5000 non-null   object        
 3   Region                5000 non-null   object        
 4   City                  5000 non-null   object        
 5   Category              5000 non-null   object        
 6   Sub-Category          5000 non-null   object        
 7   Product Name          5000 non-null   object        
 8   Quantity              5000 non-null   int64         
 9   Unit Price            5000 non-null   int64         
 10  Discount              5000 non-null   int64         
 11  Sales                 5000 non-null   float64       
 12  Profit                5000 non-null   float64       
 13  Payment Mode      

In [18]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical Columns:", categorical_cols)

Categorical Columns: ['Customer Name', 'Region', 'City', 'Category', 'Sub-Category', 'Product Name', 'Payment Mode', 'Category_SubCategory', 'Holiday_Type', 'Date_City_Key', 'Rainfall_Category', 'Season']


In [19]:
df = df.drop(columns=['Customer Name', 'Order ID'])

In [20]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

numerical_features = [
    'Year', 'Month', 'Day', 'DayOfWeek', 'IsWeekend', 'Quarter', 'Sales_Lag_1',
    'Sales_Lag_7', 'Sales_Lag_14', 'Rolling_Avg_7', 'Rolling_Avg_14', 'Is_Holiday', 'Temperature_Avg', 
    'Weekend_Holiday', 'Holiday_Rainfall'
]

categorical_features = ['Season', 'Rainfall_Category']

X = df[numerical_features + categorical_features]
y = df['Sales']

split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]


In [21]:
tscv = TimeSeriesSplit(n_splits=5)


In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

In [23]:
from xgboost import XGBRegressor
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(random_state=42, objective='reg:squarederror'))
])

In [24]:
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.8, 1.0]
}
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=tscv, 
    scoring='neg_mean_absolute_error',
    n_jobs=-1, # Uses all available CPU cores
    verbose=1  # Prints progress
)

In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

In [26]:
print("Starting Grid Search...")
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters Found: {grid_search.best_params_}")



Starting Grid Search...
Fitting 5 folds for each of 54 candidates, totalling 270 fits

Best Parameters Found: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 300, 'model__subsample': 0.8}


In [35]:
from sklearn.model_selection import TimeSeriesSplit

numerical_features = ['Quantity', 'Unit Price', 'Discount', 'Year', 'Month', 'Day', 
                      'DayOfWeek', 'IsWeekend', 'Quarter', 'Sales_Lag_7', 'Sales_Lag_14',
                      'Rolling_Avg_7', 'Rolling_Avg_14', 'Customer_Count',
                      'Is_Holiday', 'Temperature_Avg',
                      'Weekend_Holiday', 'Holiday_Rainfall']

categorical_features = ['Season', 'Rainfall_Category']

X = df[numerical_features+categorical_features]
y = df['Sales']

tscv = TimeSeriesSplit(n_splits=5)

for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

import xgboost as xgb

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(n_estimators=300, random_state=42,max_depth=3,learning_rate=0.01,subsample=0.8))
])

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)

# Evaluate
from sklearn.metrics import mean_absolute_error, mean_squared_error
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")


MAE: 8531.61
RMSE: 11056.77


In [36]:
from sklearn.metrics import mean_absolute_percentage_error

# Calculate MAPE
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
print(f"MAPE: {mape:.2f}%")


MAPE: 39.93%


In [37]:
df.to_csv("preprocessed_data.csv")

In [38]:
wape = np.sum(np.abs(y_test - y_pred)) / np.sum(y_test)
print(f"Model WAPE: {wape * 100:.2f}%")

Model WAPE: 8.26%


In [39]:
import joblib

In [40]:
joblib.dump(pipeline, 'Revenue_pipeline_best.joblib')

['Revenue_pipeline_best.joblib']

In [43]:
joblib.dump(preprocessor,'preprocessor.joblib')

['preprocessor.joblib']

In [42]:
X_train.to_csv("Train_data.csv")
X_test.to_csv("testing_data.csv")

In [44]:
!pip show scikit-learn


Name: scikit-learn
Version: 1.6.1
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License: BSD 3-Clause License

 Copyright (c) 2007-2024 The scikit-learn developers.
 All rights reserved.

 Redistribution and use in source and binary forms, with or without
 modification, are permitted provided that the following conditions are met:

 * Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

 * Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

 * Neither the name of the copyright holder nor the names of its
   contributors may be used to endorse or promote products derived from
   this software without specific prior written permission.

 THIS SOFTWARE IS PROVIDED BY THE COPYR